<a href="https://colab.research.google.com/github/tasyriqomar/ColabMD-Edu_Protein-NA/blob/main/CloudMD_Edu_Colab_Protein_NucleicAcid.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

<img src="https://raw.githubusercontent.com/tasyriqomar/ColabMD-Edu/main/Mascot_ColabMD-Edu.png" height="200" align="right" style="height:240px">

# **ColabMD-Edu: Protein-Nucleic Acid Dynamics**

Perform coarse-grained molecular dynamic simulation, convert the trajectories into all-atom, and analyse using GROMACS, PLIP and MM-GBSA. Findings are used to aid in understanding the binding interaction and energy of protein-DNA/RNA dynamics. For more details, see bottom of the notebook, checkout the ColabMD-Edu Github and publsihed works  

In [ ]:
# Install dependencies
!apt-get update
!apt-get install -y gromacs
!pip install biopython
!apt-get install python2 -y
!pip install pdbfixer openmm

Get:1 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Get:2 https://cli.github.com/packages stable InRelease [3,917 B]
Get:3 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Get:4 https://cli.github.com/packages stable/main amd64 Packages [354 B]
Hit:5 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:6 http://security.ubuntu.com/ubuntu jammy-security/universe amd64 Packages [1,307 kB]
Get:7 http://security.ubuntu.com/ubuntu jammy-security/restricted amd64 Packages [7,228 kB]
Get:8 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Get:9 http://security.ubuntu.com/ubuntu jammy-security/main amd64 Packages [4,030 kB]
Get:10 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease [18.1 kB]
Hit:11 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Get:12 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]
Get:13 http://archive.ubuntu.com/ubuntu jammy-backports InRelease [127 kB

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Create the folders
!mkdir -p "/content/drive/MyDrive/ColabMD-Edu_Protein-NA/Aptamer-Processing"

## **Step 1: Aptamer Processing**
- DNA Sequence: TCGGACATCGGATTGTCTGA
- RNA Sequence: UCGGACAUCGGAUUGUCUGA
- Predict secondary structure with [RNAfold](http://rna.tbi.univie.ac.at/cgi-bin/RNAWebSuite/RNAfold.cgi)
- Submit sequence to [RNAComposer](https://rnacomposer.cs.put.poznan.pl/) and download Aptamer.pdb

In [ ]:
import os
os.chdir('/content/drive/MyDrive/ColabMD-Edu_Protein-NA/Aptamer-Processing')

In [ ]:
# Import aptamer: xxx.pdb from downloaded folder
from google.colab import files
uploaded = files.upload()

In [ ]:
import os
os.chdir('/content/drive/MyDrive/ColabMD-Edu_Protein-NA')

In [ ]:
# 1. Clone the root repository cleanly
!git clone https://github.com/tasyriqomar/ColabMD-Edu_Protein-NA.git

# 2. Extract the specific .ff folder out to your current working directory
!cp -r ColabMD-Edu_Protein-NA/Aptamer-Processing/ .

# 3. Clean up the cloned repository folder to keep your workspace tidy
!rm -rf ColabMD-Edu_Protein-NA

# 4. Verify that both the script and the force field folder are in the same directory
!ls -d Aptamer-Processing

In [ ]:
import os
os.chdir('/content/drive/MyDrive/ColabMD-Edu_Protein-NA/Aptamer-Processing')

In [ ]:
# Convert RNA to DNA
!python rna_to_dna_with_methyl.py
!ls  # check aptamer_DNA.pdb

In [ ]:
# Rename DNA
!python rename_dna.py
!ls  # check aptamer_fixed.pdb

## **Step 2: All-atom MD Simulations of DNA Aptamer**

*   Preparation
*   Minimization
*  Equilibration
*   Run Production
*   Centered & Fitted

In [ ]:
import os
os.chdir('/content/drive/MyDrive/ColabMD-Edu_Protein-NA')

In [ ]:
# 1. Clone the root repository cleanly
!git clone https://github.com/tasyriqomar/ColabMD-Edu_Protein-NA.git

# 2. Extract the specific .ff folder out to your current working directory
!cp -r ColabMD-Edu_Protein-NA/MD/AAMD/ .

# 3. Clean up the cloned repository folder to keep your workspace tidy
!rm -rf ColabMD-Edu_Protein-NA

# 4. Verify that both the script and the force field folder are in the same directory
!ls -d AAMD

In [ ]:
# Copy the AA pdb file: !cp [source] [destination]; aptamer_fixed.pdb
!cp "/content/drive/MyDrive/ColabMD-Edu_Protein-NA/Aptamer-Processing/aptamer_fixed.pdb" "/content/drive/MyDrive/ColabMD-Edu_Protein-NA/AAMD"

In [ ]:
import os
os.chdir('/content/drive/MyDrive/ColabMD-Edu_Protein-NA/AAMD')

In [ ]:
# Convert pdb format to gromacs format (gmx)
!gmx pdb2gmx -f aptamer_fixed.pdb -o aptamer.gro -p topol.top -ff amber99sb-ildn -water tip3p -ignh

*System Prepartion*

In [ ]:
# Create box
!gmx editconf -f aptamer.gro -o boxed.gro -c -d 1.0 -bt cubic

In [ ]:
# Solvate the system
!gmx solvate -cp boxed.gro -cs spc216.gro -o solvated.gro -p topol.top

In [ ]:
# Grompp ion.tpr
!gmx grompp -f ions.mdp -c solvated.gro -p topol.top -o ions.tpr -maxwarn 1

In [ ]:
# Adding ions into the system
!gmx genion -s ions.tpr -o ionized.gro -p topol.top -neutral

*Energy Minimization*

In [ ]:
# Grompp em.tpr
!gmx grompp -f em.mdp -c ionized.gro -p topol.top -o em.tpr

In [ ]:
# Minimizing the system
!gmx mdrun -deffnm em -v

*Equilibration*

In [ ]:
# Grompp nvt.tpr
!gmx grompp -f nvt.mdp -c em.gro -r em.gro -p topol.top -o nvt.tpr

In [ ]:
# Equilibrating the system (NVT)
!gmx mdrun -deffnm nvt -v

In [ ]:
# Grompp npt.tpr
!gmx grompp -f npt.mdp -c nvt.gro -r nvt.gro -p topol.top -o npt.tpr

In [ ]:
# Equilibrating the system (NPT)
!gmx mdrun -deffnm npt -v

*Running Simulation*

In [ ]:
# Grompp md.tpr
!gmx grompp -f md.mdp -c npt.gro -p topol.top -o md.tpr

In [ ]:
# Running simulation for the system
!gmx mdrun -deffnm md -v

*Centered and Fitted Trajectories*

In [ ]:
# Generate xtc file
!gmx trjconv -f md.trr -o md.xtc

In [ ]:
# Center the trajectories
!gmx trjconv -s md.tpr -f md.xtc -o md_noPBC.xtc -pbc mol -center

In [ ]:
# Fit the trajectories
!gmx trjconv -s md.tpr -f md_noPBC.xtc -o md_fit.xtc -fit rot+trans

*Minimized and Dynamic PDB*

In [ ]:
# Get the minimized pdb
!gmx trjconv -s md.tpr -f md_fit.xtc -o aptamer_relaxed.pdb -dump 1000

In [ ]:
# Convert AA trajectories into PDB format
!gmx trjconv -s md.tpr -f md_noPBC.xtc -dt 100 -o dynamic.pdb

## **Step 3: Coarse-Grained MD (CGMD)**
- DNA preparation
- Protein preparation
- Complex preparation
- Coarse-Grained MD simulation
- Centered and fitted trajectories

In [ ]:
import os
os.chdir('/content/drive/MyDrive/ColabMD-Edu_Protein-NA')

In [ ]:
# 1. Clone the root repository cleanly
!git clone https://github.com/tasyriqomar/ColabMD-Edu_Protein-NA.git

# 2. Extract the specific .ff folder out to your current working directory
!cp -r ColabMD-Edu_Protein-NA/MD/CGMD/ .

# 3. Clean up the cloned repository folder to keep your workspace tidy
!rm -rf ColabMD-Edu_Protein-NA

# 4. Verify that both the script and the force field folder are in the same directory
!ls -d CGMD

In [ ]:
# Copy the AA pdb file: !cp [source] [destination]; aptamer_relaxed.pdb
!cp "/content/drive/MyDrive/ColabMD-Edu_Protein-NA/AAMD/aptamer_relaxed.pdb" "/content/drive/MyDrive/ColabMD-Edu_Protein-NA/CGMD"

*DNA Preparation*

*   Aptamer Coarse-grained





In [ ]:
import os
os.chdir('/content/drive/MyDrive/ColabMD-Edu_Protein-NA/CGMD')

In [ ]:
# Convert AA pdb to AA gromacs format (gro)
!gmx editconf -f aptamer_relaxed.pdb -o aptamer_aa.gro

In [ ]:
# Convert AA gro to CG pdb
!python2 martinize-dna.py -dnatype ss -f aptamer_aa.gro -o dna.top -x dna_CG.pdb

In [ ]:
# Convert CG pdb to CG gromacs format
!gmx editconf -f dna_CG.pdb -o aptamer_CG.gro

*Protein Preparation*

*   Protein Coarse-grained





In [ ]:
# Download the raw PDB file directly from the RCSB servers
!wget https://files.rcsb.org/download/XXXX.pdb

In [ ]:
from Bio.PDB import PDBParser, PPBuilder

# 1. Initialize the parser and load your PDB file
pdb_filename = "XXXX.pdb" # Change to your actual file path
parser = PDBParser(QUIET=True)
structure = parser.get_structure("protein", pdb_filename)

# 2. Isolate Chain A from the first model
model = structure[0]
if 'A' in model:
    chain_a = model['A']

    # 3. Build the polypeptide sequence
    ppb = PPBuilder()
    sequences = []

    for pp in ppb.build_peptides(chain_a):
        sequences.append(str(pp.get_sequence()))

    # Join sequences (handles cases where there are resolved gaps/loops)
    full_sequence = "".join(sequences)

    print("✨ Chain A Sequence (1-Letter Code):")
    print(full_sequence)
    print(f"\nTotal Residues: {len(full_sequence)}")
else:
    print("❌ Error: Chain A not found in this PDB file.")

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sokrypton/ColabFold/blob/main/AlphaFold2.ipynb)

In [ ]:
# Import protein: xxx.pdb from downloaded folder
from google.colab import files
uploaded = files.upload()

In [ ]:
# Rename the sequence
!gmx editconf -f xxx.pdb -o fixed.pdb -resnr XX

In [ ]:
input_file = "XXX.pdb"
output_file = "XXX-X.pdb"

with open(input_file, "r") as infile, open(output_file, "w") as outfile:
    for line in infile:
        # Check if the line contains atom coordinates
        if line.startswith("ATOM") or line.startswith("HETATM"):
            # Column 22 in a PDB corresponds to string index 21
            if line[21] == "A":
                # Swap character at index 21 to 'B'
                line = line[:21] + "B" + line[22:]
        outfile.write(line)

print("Chain A successfully changed to Chain B!")

In [ ]:
# Install depedencies
!sudo apt-get install dssp

In [ ]:
# Martini2 protein parameterize (Coarse-grained)
!python2 martinize.py -f XXX-X.pdb -o system.top -x CG.pdb -p backbone -ff martini22 -elastic

In [ ]:
# Generate .gro file
!gmx editconf -f CG.pdb -o CG.gro

In [ ]:
# Update the topol.top with current martini2 itp files
!sed -i 's/#include "martini.itp"/#include "martini_v2.1P-dna.itp"\n#include "martini_v2.0_ions.itp"/' system.top

*Complex Preparation*

*   Protein-DNA Coarse-grained





In [ ]:
# Insert CG DNA into CG protein
!gmx insert-molecules -f CG.gro -ci aptamer_CG.gro -box 10 10 10 -nmol 1 -o CG_aptamer.gro

In [ ]:
# Make a box and solvate the system
!python2 insane.py -f CG_aptamer.gro -o system.gro -pbc cubic -box 10,10,10 -salt 0.15 -charge auto -sol W

In [ ]:
# Update the naming
!sed -i 's/NA+/NA/g; s/CL-/CL/g' system.top

*Coarse-Grained MD Simulation*

In [ ]:
# Grompp minimization.tpr file
!gmx grompp -f minimization.mdp -c system.gro -p system.top -o minimization.tpr -maxwarn 10

In [ ]:
# Run minimization simulation
!gmx mdrun -deffnm minimization -v

In [ ]:
# Make index file
!gmx make_ndx -f minimization.gro -o index.ndx

In [ ]:
# Grompp equlibration.tpr file
!gmx grompp -f equilibration.mdp -c minimization.gro -p system.top -o equilibration.tpr -maxwarn 5 -n index.ndx

In [ ]:
# Run equilibration simulation
!gmx mdrun -deffnm equilibration -v

In [ ]:
# Grompp dynamic.tpr file
!gmx grompp -f dynamic.mdp -c equilibration.gro -p system.top -o dynamic.tpr -maxwarn 5 -n index.ndx

In [ ]:
# Run dynamic simulation
!gmx mdrun -deffnm dynamic -v

*Centered and Fitted Trajectories*

In [ ]:
# Make new index (1|12)
!gmx make_ndx -f dynamic.gro -o index.ndx

In [ ]:
# Center the trajectories
!gmx trjconv -pbc mol -center -ur compact -s dynamic.tpr -n index.ndx -f dynamic.xtc -o dynamic_center.xtc

In [ ]:
# Make new index; BB
!echo -e "a BB\nq" | gmx make_ndx -f equilibration.gro -n index.ndx

In [ ]:
# Fit the trajectories
!gmx trjconv -fit rot+trans -s dynamic.tpr -f dynamic_center.xtc -o dynamic_fitted.xtc -n index.ndx

In [ ]:
# Make multiframe pdb
!gmx trjconv -s dynamic.tpr -n index.ndx -f dynamic_center.xtc -dt 1000 -o dynamic.pdb

## **Step 4: Backmapping**

- Backmapping CG to all-atom trajectories

In [ ]:
import os
os.chdir('/content/drive/MyDrive/CloudMD-Edu_Protein-NA')

In [ ]:
# 1. Clone the root repository cleanly
!git clone https://github.com/tasyriqomar/ColabMD-Edu_Protein-NA.git

# 2. Extract the specific .ff folder out to your current working directory
!cp -r ColabMD-Edu_Protein-NA/Backmapping/ .

# 3. Clean up the cloned repository folder to keep your workspace tidy
!rm -rf ColabMD-Edu_Protein-NA

# 4. Verify that both the script and the force field folder are in the same directory
!ls -d Backmapping

In [ ]:
import os
os.chdir('/content/drive/MyDrive/CloudMD-Edu_Protein-NA/Backmapping')

*Complex All-Atom Preparation (Gromacs Format)*

In [ ]:
# Copy the CGMD files: !cp [source] [destination]; dynamic_fitted.xtc, dynamic.tpr, dynamic.mdp, index.ndx
!cp "/content/drive/MyDrive/ColabMD-Edu_Protein-NA/CGMD/dynamic_fitted.xtc" "/content/drive/MyDrive/CloudMD-Edu_Protein-NA/Backmapping"
!cp "/content/drive/MyDrive/ColabMD-Edu_Protein-NA/CGMD/dynamic.tpr" "/content/drive/MyDrive/CloudMD-Edu_Protein-NA/Backmapping"
!cp "/content/drive/MyDrive/ColabMD-Edu_Protein-NA/CGMD/index.ndx" "/content/drive/MyDrive/CloudMD-Edu_Protein-NA/Backmapping"
!cp "/content/drive/MyDrive/ColabMD-Edu_Protein-NA/CGMD/dynamic.mdp" "/content/drive/MyDrive/CloudMD-Edu_Protein-NA/Backmapping"
!cp "/content/drive/MyDrive/ColabMD-Edu_Protein-NA/CGMD/fixed.pdb" "/content/drive/MyDrive/CloudMD-Edu_Protein-NA/Backmapping"
!cp "/content/drive/MyDrive/ColabMD-Edu_Protein-NA/CGMD/aptamer_relaxed.pdb" "/content/drive/MyDrive/CloudMD-Edu_Protein-NA/Backmapping"

In [ ]:
# 1. Combine the files while filtering out 'ENDMDL' and 'END' lines
!cat RBD-B.pdb aptamer_relaxed.pdb | grep -v -E "TITLE|MODEL|ENDMDL|CRYST1|REMARK|END" > combined.pdb

# 2. Add a single, clean 'END' tag at the very bottom to properly close the PDB file
!echo "END" >> combined.pdb

In [ ]:
# Convert to Gromacs file with topology files
!gmx pdb2gmx -f combined.pdb -o aa.gro -ignh

In [ ]:
# Access the permission for the scripts
!chmod +x Dr_Tasyriq_V4_M2.sh Dr_Tasyriq_V4_M2_Redo.sh initram-v5.sh backward.py

In [ ]:
import os
os.chdir('/content/drive/MyDrive/CloudMD-Edu_Protein-NA/Backmapping/Mapping')

In [ ]:
# Access the permission for the scripts
!chmod +x __init__.py

In [ ]:
import os
os.chdir('/content/drive/MyDrive/CloudMD-Edu_Protein-NA/Backmapping')

In [ ]:
# Execute the script for looping process
!bash Dr_Tasyriq_V4_M2.sh

*All-atomic file generation*

- remove unnessary files
- concatenate the trajectories
- generate the AA dynamic.tpr
- generate the AA index.ndx

In [ ]:
# Delete ALL files ending in ns.xtc in a specific folder
!rm "/content/drive/MyDrive/ColabMD-Edu_Protein-NA/Backmapping/"*ns.xtc

In [ ]:
# Delete ALL files ending in ns.gro in a specific folder
!rm "/content/drive/MyDrive/ColabMD-Edu_Protein-NA/Backmapping/"*ns.gro

In [ ]:
# Delete ALL files ending in # in a specific folder
!rm "/content/drive/MyDrive/ColabMD-Edu_Protein-NA/Backmapping/"*#

In [ ]:
import os
os.chdir('/content/drive/MyDrive/ColabMD-Edu_Protein-NA/Backmapping/all_xtc')

In [ ]:
# Concatenate the xtc trajectories files
!gmx trjcat -f *.xtc -o all.xtc

In [ ]:
# Copy the file: !cp [source] [destination]
!cp "/content/drive/MyDrive/ColabMD-Edu_Protein-NA/Backmapping/gro/aa_0ns.gro" "/content/drive/MyDrive/CloudMD-Edu_Protein-NA/Backmapping"

In [ ]:
import os
os.chdir('/content/drive/MyDrive/ColabMD-Edu_Protein-NA/Backmapping')

In [ ]:
# AA .tpr file
!gmx grompp -f dynamic.mdp -c aa_0ns.gro -p topol.top -o dynamic.tpr -maxwarn 5

In [ ]:
# Copy the file: !cp [source] [destination]
!cp "/content/drive/MyDrive/ColabMD-Edu_Protein-NA/Backmapping/dynamic.tpr" "/content/drive/MyDrive/CloudMD-Edu_Protein-NA/Backmapping/all_xtc"

In [ ]:
import os
os.chdir('/content/drive/MyDrive/ColabMD-Edu_Protein-NA/Backmapping/all_xtc')

In [ ]:
# AA .ndx file (1|12)
!gmx make_ndx -f dynamic.tpr -o index.ndx

In [ ]:
# Create multiple frames of PDB file
!gmx trjconv -s dynamic.tpr -n index.ndx -f all.xtc -dt 1000 -o concatenated.pdb

## **Step 5: Analysis**
- Trajectory processing, PyMOL visualization, PLIP interactions, MM-GBSA energy evaluation, PCA-FEL.

In [ ]:
# Create the folder first
!mkdir -p "/content/drive/MyDrive/ColabMD-Edu_Protein-NA/Analysis"

# Copy the AAMD file: !cp [source] [destination]
!cp "/content/drive/MyDrive/ColabMD-Edu_Protein-NA/Backmapping/all_xtc/all.xtc" "/content/drive/MyDrive/ColabMD-Edu_Protein-NA/Analysis"
!cp "/content/drive/MyDrive/ColabMD-Edu_Protein-NA/Backmapping/all_xtc/dynamic.tpr" "/content/drive/MyDrive/ColabMD-Edu_Protein-NA/Analysis"
!cp "/content/drive/MyDrive/ColabMD-Edu_Protein-NA/Backmapping/all_xtc/index.ndx" "/content/drive/MyDrive/ColabMD-Edu_Protein-NA/Analysis"
!cp "/content/drive/MyDrive/ColabMD-Edu_Protein-NA/Backmapping/all_xtc/concatenated.pdb" "/content/drive/MyDrive/ColabMD-Edu_Protein-NA/Analysis"

In [ ]:
import os
os.chdir('/content/drive/MyDrive/ColabMD-Edu_Protein-NA/Analysis')

## *Step 5.1: PLIP*

- Installation
- Running
- Figure Generation

**PLIP Installation**

In [ ]:
# Install conda in Colab 3.12 environment
!pip install -q condacolab
import condacolab
condacolab.install()

In [ ]:
# This forces the installation of the libraries into the 3.12 environment
!mamba install -c conda-forge openbabel plip pymol-open-source --no-pin -y

In [ ]:
# This forces the use of the 3.11 environemnt for PLIP and openbabel
!rm /usr/local/conda-meta/pinned
!mamba install -c conda-forge openbabel plip pymol-open-source -y

In [ ]:
# Install of depedencies for the figure generations
!mamba install -c conda-forge matplotlib pandas seaborn openpyxl --yes

**PLIP Execution**

In [ ]:
import os
os.chdir('/content/drive/MyDrive/ColabMD-Edu_Protein-NA')

In [ ]:
# 1. Clone the root repository cleanly
!git clone https://github.com/tasyriqomar/ColabMD-Edu_Protein-NA.git

# 2. Extract the specific CGMD folder out to your current working directory
!cp -r ColabMD-Edu_Protein-NA/Analysis .

# 3. Clean up the cloned repository folder to keep your workspace tidy
!rm -rf ColabMD-Edu_Protein-NA

# 4. Verify that both the script and the force field folder are in the same directory
!ls -d Analysis*

In [ ]:
import os
os.chdir('/content/drive/MyDrive/ColabMD-Edu_Protein-NA/Analysis')

In [ ]:
# Split the concatenated.pdb
!python split_pdb.py

In [ ]:
# Change directory to split_pdbs
import os
os.chdir('/content/drive/MyDrive/ColabMD-Edu_Protein-Protein/NA/split_pdbs')

In [ ]:
# Copy the file: !cp [source] [destination]
!cp "/content/drive/MyDrive/ColabMD-Edu_Protein-NA/Analysis/plip.sh" "/content/drive/MyDrive/ColabMD-Edu_Protein-NA/Analysis/split_pdbs"
!cp "/content/drive/MyDrive/ColabMD-Edu_Protein-NA/Analysis/convert_xml_to_json.py" "/content/drive/MyDrive/ColabMD-Edu_Protein-NA/Analysis/split_pdbs"
!cp "/content/drive/MyDrive/ColabMD-Edu_Protein-NA/Analysis/process_json.py" "/content/drive/MyDrive/ColabMD-Edu_Protein-NA/Analysis/split_pdbs"
!cp "/content/drive/MyDrive/ColabMD-Edu_Protein-NA/Analysis/type_interactions_color.py" "/content/drive/MyDrive/ColabMD-Edu_Protein-NA/Analysis/split_pdbs"
!cp "/content/drive/MyDrive/ColabMD-Edu_Protein-NA/Analysis/residue_interactions_color_csv_a.py" "/content/drive/MyDrive/ColabMD-Edu_Protein-NA/Analysis/split_pdbs"
!cp "/content/drive/MyDrive/ColabMD-Edu_Protein-NA/Analysis/timeline_interaction_color.py" "/content/drive/MyDrive/ColabMD-Edu_Protein-NA/Analysis/split_pdbs"

In [ ]:
# Run bash file that execute the PLIP
!bash plip.sh

In [ ]:
# Convert *.xml files to a .json file
!python convert_xml_to_json.py

In [ ]:
# Generate folders (CSV) for different type of interaction
!python process_json.py

**Generate PLIP figures**

In [ ]:
# Figure Percentage of Interaction
!MPLBACKEND=Agg python /content/drive/MyDrive/ColabMD-Edu_Protein-NA/Analysis/split_pdbs/percentage_interactions.py

In [ ]:
# Figure Type of interaction
!MPLBACKEND=Agg python /content/drive/MyDrive/ColabMD-Edu_Protein-NA/Analysis/split_pdbs/type_interactions_color.py

In [ ]:
# Figure Residue of interaction
!MPLBACKEND=Agg python /content/drive/MyDrive/ColabMD-Edu_Protein-NA/Analysis/split_pdbs/residue_interactions_color_csv_a.py

In [ ]:
# Figure Timeline of interaction
!MPLBACKEND=Agg python /content/drive/MyDrive/ColabMD-Edu_Protein-NA/Analysis/split_pdbs/timeline_interaction_color.py

## *Step 5.2: MM-GBSA*

- Installation
- Running
- Figure Generation

**MM-GBSA Installation**

In [ ]:
# Create the folder first
!mkdir -p "/content/drive/MyDrive/ColabMD-Edu_Protein-NA/Analysis/MMGBSA"

# Copy the file: !cp [source] [destination]
!cp "/content/drive/MyDrive/ColabMD-Edu_Protein-NA/Analysis/all.xtc" "/content/drive/MyDrive/ColabMD-Edu_Protein-NA/Analysis/MMGBSA"
!cp "/content/drive/MyDrive/ColabMD-Edu_Protein-NA/Analysis/dynamic.tpr" "/content/drive/MyDrive/ColabMD-Edu_Protein-NA/Analysis/MMGBSA"
!cp "/content/drive/MyDrive/ColabMD-Edu_Protein-NA/Analysis/index.ndx" "/content/drive/MyDrive/ColabMD-Edu_Protein-NA/Analysis/MMGBSA"
!cp "/content/drive/MyDrive/ColabMD-Edu_Protein-NA/Backmapping/aa.gro" "/content/drive/MyDrive/ColabMD-Edu_Protein-NA/Analysis/MMGBSA"
!cp "/content/drive/MyDrive/ColabMD-Edu_Protein-NA/Backmapping/topol.top" "/content/drive/MyDrive/ColabMD-Edu_Protein-NA/Analysis/MMGBSA"
!cp "/content/drive/MyDrive/ColabMD-Edu_Protein-NA/Backmapping/posre_DNA.itp" "/content/drive/MyDrive/ColabMD-Edu_Protein-NA/Analysis/MMGBSA"
!cp "/content/drive/MyDrive/ColabMD-Edu_Protein-NA/Backmapping/posre_Protein_chain_B.itp" "/content/drive/MyDrive/ColabMD-Edu_Protein-NA/Analysis/MMGBSA"
!cp "/content/drive/MyDrive/ColabMD-Edu_Protein-NA/Backmapping/topol_DNA.itp" "/content/drive/MyDrive/ColabMD-Edu_Protein-NA/Analysis/MMGBSA"
!cp "/content/drive/MyDrive/ColabMD-Edu_Protein-NA/Backmapping/topol_Protein_chain_B.itp" "/content/drive/MyDrive/ColabMD-Edu_Protein-NA/Analysis/MMGBSA"

In [ ]:
# Copy the folder: !cp -r /path/to/source_folder /path/to/destination_folder
!cp -r "/content/drive/MyDrive/ColabMD-Edu_Protein-NA/Backmapping/Mapping/" "/content/drive/MyDrive/ColabMD-Edu_Protein-NA/Analysis/MMGBSA"


In [ ]:
import os
os.chdir('/content/drive/MyDrive/ColabMD-Edu_Protein-NA/Analysis/MMGBSA')

In [ ]:
# Obtain mmpbsa.in
!wget https://raw.githubusercontent.com/tasyriqomar/ColabMD-Edu_Protein-NA/refs/heads/main/Analysis/mmpbsa.in

In [ ]:
# Install conda in Colab 3.12 environment
!pip install -q condacolab
import condacolab
condacolab.install()

In [ ]:
# Install AmberTools, MPI, and gmx_MMPBSA from the official channel
!mamba install -c conda-forge ambertools mpi4py gmx_mmpbsa -y

In [ ]:
# This forces the use of the 3.11 environemnt for ambertools and gmx_mmpbsa
!rm /usr/local/conda-meta/pinned
!mamba install -c conda-forge ambertools mpi4py gmx_mmpbsa -y

**MM-GBSA Execution**

In [ ]:
# Run MM-PBSA between Sert protein (1) and DNA (12)
!gmx_MMPBSA -O -i mmpbsa.in -cs dynamic.tpr -ct all.xtc -ci index.ndx -cg 1 12 -cp topol.top -o FINAL_RESULTS_MMPBSA.dat -eo FINAL_RESULTS_MMPBSA.csv -do FINAL_DECOMP_MMPBSA.dat -deo FINAL_DECOMP_MMPBSA.csv

**Generate MM-GBSA table & figures**

In [ ]:
# Delta (Complex-receptor-ligand)
!tail -n 20 FINAL_RESULTS_MMPBSA.dat

In [ ]:
# Obtain per-residue.py
!wget https://raw.githubusercontent.com/tasyriqomar/ColabMD-Edu_Protein-NA/refs/heads/main/Analysis/per-residue.py

In [ ]:
# Generate decomposition bar chart
!python per-residue.py

In [ ]:
# Obtain heatmap.py
!wget https://raw.githubusercontent.com/tasyriqomar/ColabMD-Edu_Protein-NA/refs/heads/main/Analysis/heatmap.py

In [ ]:
# Generate heatmap
!python heatmap.py

## *Step 5.3: PCA and FEL*

- Installation
- Running
- Figure Generation

**PCA-FEL Installation**

In [ ]:
# Create the folder first
!mkdir -p "/content/drive/MyDrive/ColabMD-Edu_Protein-NA/Analysis/PCA-FEL"

# Copy the file: !cp [source] [destination]
!cp "/content/drive/MyDrive/ColabMD-Edu_Protein-NA/Analysis/all.xtc" "/content/drive/MyDrive/ColabMD-Edu_Protein-NA/Analysis/PCA-FEL"
!cp "/content/drive/MyDrive/ColabMD-Edu_Protein-NA/Analysis/dynamic.tpr" "/content/drive/MyDrive/ColabMD-Edu_Protein-NA/Analysis/PCA-FEL"
!cp "/content/drive/MyDrive/ColabMD-Edu_Protein-NA/Analysis/index.ndx" "/content/drive/MyDrive/ColabMD-Edu_Protein-NA/Analysis/PCA-FEL"


In [ ]:
import os
os.chdir('/content/drive/MyDrive/ColabMD-Edu_Protein-NA/Analysis/PCA-FEL')

**PCA Execution**

In [ ]:
# Covariance: least square fir (Backbone) and covariance analysis (Backbone)
!gmx covar -f all.xtc -s dynamic.tpr -n index.ndx -o eigenval.xvg -v eigenvec.trr -av average.pdb -l covar.log -b 0 -tu ns

In [ ]:
# Eigenvalue: least square fit (Backbone) and component eigenvector (Backbone)
!gmx anaeig -v eigenvec.trr -f all.xtc -s dynamic.tpr -n index.ndx -comp eigcomp1-2.xvg -rmsf eigrmsf1-2.xvg -2d 2dproj1-2.xvg -b 0 -tu ns -first 1 -last 2
!gmx anaeig -v eigenvec.trr -f all.xtc -s dynamic.tpr -n index.ndx -comp eigcomp1-3.xvg -rmsf eigrmsf1-3.xvg -2d 2dproj1-3.xvg -b 0 -tu ns -first 1 -last 3
!gmx anaeig -v eigenvec.trr -f all.xtc -s dynamic.tpr -n index.ndx -comp eigcomp2-3.xvg -rmsf eigrmsf2-3.xvg -2d 2dproj2-3.xvg -b 0 -tu ns -first 2 -last 3

**FEL Execution**

In [ ]:
# Sham:
!gmx sham -f 2dproj1-2.xvg -notime -ls gibb1-2.xpm

In [ ]:
# Obtain per-residue.py
!wget https://raw.githubusercontent.com/tasyriqomar/ColabMD-Edu_Protein-NA/refs/heads/main/Analysis/xpm2txt.py

In [ ]:
!python2 xpm2txt.py -f gibb1-2.xpm -o FEL.dat

**Generate PCA & FEL figures**

In [ ]:
# Obtain per-residue.py
!wget https://raw.githubusercontent.com/tasyriqomar/ColabMD-Edu_Protein-NA/refs/heads/main/Analysis/PCA.py

In [ ]:
# Generate PCA plot
!python PCA.py

In [ ]:
# Obtain per-residue.py
!wget https://raw.githubusercontent.com/tasyriqomar/ColabMD-Edu_Protein-NA/refs/heads/main/Analysis/FEL.py

In [ ]:
# Generate FEL plot
!python FEL.py

## *Step 5.4: PyMOL*

- Installation
- Running
- Video Generation

**PyMOL Installation**

In [ ]:
# 1. Force-delete the broken system folder causing the '_cmd' error
!rm -rf /usr/lib/python3/dist-packages/pymol

# 2. Install and launch the Conda handler
!pip install -q condacolab
import condacolab
condacolab.install()

In [ ]:
import os
# Clear out background blocks
if os.path.exists("/usr/local/conda-meta/pinned"):
    os.remove("/usr/local/conda-meta/pinned")

# Install working PyMOL and image tools via mamba
!mamba install -y -c conda-forge "python=3.12" "pymol-open-source" imageio

**PyMOL Execution**

In [ ]:
import sys
import os
import glob

# Path Finder Fix
conda_site_packages = glob.glob("/usr/local/lib/python3.*/site-packages")
for path in conda_site_packages:
    if path not in sys.path:
        sys.path.insert(0, path)

import pymol
from pymol import cmd
import imageio
from IPython.display import Image, display

# 1. Open the headless PyMOL backend
pymol.finish_launching(['pymol', '-cq'])

# Optimize RAM usage for heavy all-atom PDB files
cmd.set("defer_builds_mode", 3)

# 2. Specify your file name
pdb_trajectory = "concatenated.pdb"
object_name = "aptamer_complex"

print(f"Opening trajectory: {pdb_trajectory}...")
cmd.load(pdb_trajectory, object_name)

# 3. Verify how many frames were loaded
num_frames = cmd.count_states(object_name)
print(f"Loaded {num_frames} trajectory frames.")

# 4. Explicit Selections
protein_sel = f"{object_name} and resn ALA+ARG+ASN+ASP+CYS+GLN+GLU+GLY+HIS+ILE+LEU+LYS+MET+PHE+PRO+SER+THR+TRP+TYR+VAL"
dna_sel = f"{object_name} and resn DA+DC+DG+DT+A+C+G+T+U"

cmd.hide("everything")
cmd.show("ribbon", protein_sel)
cmd.show("cartoon", dna_sel)

# Color-code
cmd.color("marine", protein_sel)
cmd.color("orange", dna_sel)
cmd.zoom()

# 5. Structural Alignment
print("Aligning trajectory frames to prevent drifting...")
cmd.intra_fit(f"{object_name} and name CA", 1)

# 6. Render the frames to create a movie
frame_step = 1  # Renders all 22 frames seamlessly
image_files = []

print("Rendering frames to temporary images...")
os.makedirs("trajectory_movie", exist_ok=True)

for state in range(1, num_frames + 1, frame_step):
    img_path = f"trajectory_movie/frame_{state:04d}.png"

    # FIX: Explicitly switch PyMOL to the target frame before taking the PNG
    cmd.frame(state)
    cmd.png(img_path, width=600, height=600, ray=0)

    image_files.append(img_path)

# 7. Compile the images into a looping animated GIF
print("Compiling frames into an animated GIF...")
gif_output = "trajectory_animation.gif"
with imageio.get_writer(gif_output, mode='I', duration=0.1, loop=0) as writer:
    for filename in image_files:
        image = imageio.imread(filename)
        writer.append_data(image)

print("\nProcessing complete! Here is your trajectory movie:")
display(Image(filename=gif_output))